In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Paths

In [3]:
BASE_DIR = "/content/drive/MyDrive/genai_project"

CLEAN_DIR = f"{BASE_DIR}/data/synthetic_clean"
DEGRADED_DIR = f"{BASE_DIR}/data/synthetic_degraded"
REAL_DIR = f"{BASE_DIR}/data/real"

TARGET_DIR = f"{BASE_DIR}/data/split"
REAL_TEST_DIR = f"{BASE_DIR}/data/real_test"

Reset

In [4]:
import shutil, os

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

if os.path.exists(REAL_TEST_DIR):
    shutil.rmtree(REAL_TEST_DIR)

print("Reset folders")

Reset folders


Collect function

In [5]:
def collect_images(base_path):
    data = {}

    for cls in os.listdir(base_path):
        class_path = os.path.join(base_path, cls)
        if not os.path.isdir(class_path):
            continue

        imgs = [os.path.join(class_path, f) for f in os.listdir(class_path)]
        data[cls] = imgs

    return data

Load data

In [6]:
import random
random.seed(42)

clean_data = collect_images(CLEAN_DIR)
degraded_data = collect_images(DEGRADED_DIR)
real_data = collect_images(REAL_DIR)

create train dataset

In [7]:
data = {}

for cls in degraded_data:

    degraded = degraded_data.get(cls, [])
    clean = clean_data.get(cls, [])

    random.shuffle(degraded)
    random.shuffle(clean)

    # ratio
    n_deg = int(0.7 * len(degraded))
    n_clean = int(0.3 * len(clean))

    selected = random.sample(degraded, min(len(degraded), n_deg)) + random.sample(clean, min(len(clean), n_clean))

    random.shuffle(selected)

    data[cls] = selected

Split synthetic (train/val/test)

In [8]:
train_ratio = 0.7
val_ratio = 0.15

split_counts = {}

for cls, images in data.items():

    random.shuffle(images)

    n = len(images)
    n_train = int(train_ratio * n)
    n_val = int(val_ratio * n)

    splits = {
        "train": images[:n_train],
        "val": images[n_train:n_train+n_val],
        "test": images[n_train+n_val:]
    }

    split_counts[cls] = {}

    for split in splits:
        os.makedirs(f"{TARGET_DIR}/{split}/{cls}", exist_ok=True)

        for img_path in splits[split]:
            filename = os.path.basename(img_path)
            dst = f"{TARGET_DIR}/{split}/{cls}/{filename}"
            shutil.copy(img_path, dst)

        split_counts[cls][split] = len(splits[split])

real test

In [9]:
for cls in real_data:

    os.makedirs(f"{REAL_TEST_DIR}/{cls}", exist_ok=True)

    for img_path in real_data[cls]:
        filename = os.path.basename(img_path)
        dst = f"{REAL_TEST_DIR}/{cls}/{filename}"
        shutil.copy(img_path, dst)

print summary

In [10]:
print("\nSynthetic split:\n")

for cls in split_counts:
    print(f"{cls}:")
    for split in split_counts[cls]:
        print(f"  {split}: {split_counts[cls][split]}")
    print()

print("\nReal test set:")
for cls in real_data:
    print(f"{cls}: {len(real_data[cls])}")


Synthetic split:

clean:
  train: 70
  val: 15
  test: 15

empty:
  train: 72
  val: 15
  test: 16

finished_leftovers:
  train: 70
  val: 15
  test: 15

full:
  train: 101
  val: 21
  test: 23


Real test set:
full: 31
empty: 11
clean: 5
finished_leftovers: 18
